# 🧬 Banco de Variantes Genéticas de BRCA1 e BRCA2

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alinesmdados/banco-variantes-brca/blob/main/banco_variantes_brca.ipynb)

Este notebook constrói um **banco de dados relacional (SQLite)** com as variantes genéticas dos genes **BRCA1** e **BRCA2** registradas no **ClinVar**, o banco público do NCBI que reúne a interpretação clínica de variantes humanas. Em seguida, responde perguntas com **SQL** e apresenta os resultados em um **dashboard**.

Variantes patogênicas em BRCA1/BRCA2 aumentam significativamente o risco de câncer de mama e de ovário hereditários. Saber **quais variantes já foram classificadas, com que grau de evidência e quantas ainda têm significado incerto** é central para a interpretação de exames genéticos.

**Sumário**
1. ETL: extração dos dados do ClinVar e carga no banco
2. Modelo de dados
3. Consultas SQL
4. Dashboard
5. Consulta livre
6. Considerações sobre proteção de dados (LGPD)

> Para rodar: *Ambiente de execução → Executar tudo*. O download dos dados leva de 1 a 3 minutos.

In [15]:
# Configuração: baixa o código do repositório (no Colab) e importa as dependências
import os

if not os.path.exists("etl_clinvar.py"):
    !git clone -q https://github.com/alinesmdados/banco-variantes-brca.git
    %cd banco-variantes-brca

import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import etl_clinvar as etl
import dashboard as dash

pd.set_option("display.max_colwidth", 80)
print("Pronto ✔")

Pronto ✔


## 1. ETL: extração, transformação e carga

O módulo `etl_clinvar.py` executa o pipeline:

| Etapa | O que acontece |
|---|---|
| **Extração** | Busca no ClinVar, pela API pública *E-utilities* do NCBI, as variantes que afetam **apenas** BRCA1 ou BRCA2 (`esearch`) e baixa seus resumos em lotes (`esummary`, formato JSON) |
| **Transformação** | Agrupa as classificações clínicas em 5 categorias, converte o status de revisão em **estrelas** (0 a 4), padroniza datas, extrai coordenadas no genoma de referência GRCh38 e as condições associadas |
| **Carga** | Insere os registros no banco SQLite modelado em `schema.sql`, respeitando chaves primárias e estrangeiras |

In [ ]:
conn = etl.construir_banco("clinvar_brca.db", genes=["BRCA1", "BRCA2"])

BRCA1: 14.132 variantes encontradas no ClinVar. Baixando...


## 2. Modelo de dados

```
genes (1) ──< variantes (N) >── (1) classificacoes
                   │
                   └──< variante_condicao >── condicoes      (relação N:N)
```

- **genes**: BRCA1 e BRCA2, com o ID do NCBI Gene e o cromossomo.
- **classificacoes**: texto original do ClinVar (ex.: *Likely pathogenic*) e a **categoria** usada nas análises.
- **variantes**: uma linha por variante, com tipo, consequência molecular, alteração na proteína, posição, estrelas e data da última avaliação.
- **condicoes** e **variante_condicao**: doenças associadas a cada variante (uma variante pode estar ligada a várias condições).
- **vw_variantes**: *view* que junta as tabelas e simplifica as consultas.

In [ ]:
tabelas = ["genes", "classificacoes", "variantes", "condicoes", "variante_condicao"]
display(pd.DataFrame({
    "Tabela": tabelas,
    "Registros": [conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in tabelas],
}))

print("Amostra da view vw_variantes:")
display(pd.read_sql_query("SELECT * FROM vw_variantes LIMIT 5", conn))

## 3. Consultas SQL

Todas as consultas estão no arquivo `consultas.sql`. A função `mostrar` exibe o código SQL e o resultado.

In [ ]:
from IPython.display import Markdown

def mostrar(nome):
    display(Markdown(f"```sql\n{dash.CONSULTAS[nome]}\n```"))
    display(dash.consultar(conn, nome))

### 3.1 Visão geral: quantas variantes e qual a proporção de patogênicas e incertas?

In [ ]:
mostrar("visao_geral")

### 3.2 Distribuição das categorias por gene

In [ ]:
mostrar("categorias_por_gene")

### 3.3 Tipos de consequência mais comuns entre as variantes patogênicas

Usa a **função de janela** `RANK() OVER (PARTITION BY gene ...)` para ranquear as consequências dentro de cada gene.

In [ ]:
mostrar("consequencias_patogenicas")

### 3.4 Qual a chance de uma variante ser patogênica, de acordo com sua consequência?

Considera apenas variantes já classificadas (patogênicas ou benignas). Espera-se que variantes que **truncam a proteína**, como *frameshift* e *nonsense*, tenham taxas muito maiores que variantes *missense* ou sinônimas.

In [ ]:
mostrar("taxa_patogenicidade_por_consequencia")

### 3.5 Qualidade da evidência

O ClinVar atribui **estrelas** conforme o nível de revisão: 0 (sem critério), 1 (um laboratório com critérios), 2 (vários laboratórios concordantes), 3 (painel de especialistas) e 4 (diretriz clínica).

In [ ]:
mostrar("qualidade_evidencia")

### 3.6 Condições mais associadas às variantes patogênicas

In [ ]:
mostrar("condicoes_mais_citadas")

### 3.7 Exemplos de variantes patogênicas revisadas por painel de especialistas

In [ ]:
mostrar("variantes_expert_panel")

## 4. Dashboard

Painel com os principais indicadores, gerado pela função `dashboard.painel` e salvo também como `painel_brca.png`.

In [ ]:
fig = dash.painel(conn, salvar_em="painel_brca.png")
plt.show()

## 5. Consulta livre

Escreva sua própria consulta SQL na célula abaixo. Sugestão: troque `'BRCA1'` por `'BRCA2'` ou filtre outra categoria.

In [ ]:
minha_consulta = """
SELECT consequencia, COUNT(*) AS variantes
FROM vw_variantes
WHERE gene = 'BRCA1' AND categoria = 'Significado incerto'
GROUP BY consequencia
ORDER BY variantes DESC
LIMIT 10
"""
pd.read_sql_query(minha_consulta, conn)

Para baixar o banco e usá-lo em outra ferramenta (DBeaver, Power BI, Metabase etc.), execute a célula abaixo no Colab.

In [ ]:
try:
    from google.colab import files
    files.download("clinvar_brca.db")
except ImportError:
    print("Fora do Colab: o arquivo clinvar_brca.db está na pasta do projeto.")

## 6. Considerações sobre proteção de dados (LGPD)

A Lei Geral de Proteção de Dados (Lei nº 13.709/2018) classifica o **dado genético** como **dado pessoal sensível** (art. 5º, II), sujeito a regras mais rígidas de tratamento (art. 11).

Este projeto utiliza apenas dados **públicos e agregados** do ClinVar: cada registro descreve uma **variante** e sua interpretação clínica, não um indivíduo. Não há nomes, identificadores nem genótipos de pacientes.

Em um cenário real, por exemplo um laboratório que cruzasse os exames de seus pacientes com este banco, seriam necessários, entre outros cuidados: base legal adequada (como a tutela da saúde), controle de acesso, registro das operações de tratamento, pseudonimização dos dados dos pacientes e relatório de impacto à proteção de dados pessoais.

## Conclusões

- O pipeline monta, de forma **reprodutível**, um banco relacional atualizado a partir de uma fonte pública oficial, com dados que mudam conforme novas evidências são publicadas.
- As consultas SQL permitem responder perguntas clínicas relevantes, como a proporção de **variantes de significado incerto**, um dos principais desafios da genética clínica, e a relação entre o **tipo de alteração molecular** e a **patogenicidade**.
- O código é coberto por **testes automatizados** (`pytest`) no arquivo `test_banco.py`.

**Próximos passos possíveis:** incluir outros genes de predisposição ao câncer (PALB2, TP53, ATM), acompanhar reclassificações entre versões do ClinVar e publicar o dashboard em uma ferramenta de BI.